In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import urllib.request, zipfile, io
!pip install -q gdown
import gdown
import os

In [ ]:
try:
  drive.mount("/content/drive")
  os.makedirs("/content/drive/MyDrive/Intro_ML_Summative/results", exist_ok=True)
  RESULTS_PATH = "/content/drive/MyDrive/Intro_ML_Summative/results/diabetes_results.csv"
  MODEL_DIR = "/content/drive/MyDrive/Intro_ML_Summative/models"
  FIG_DIR = "/content/drive/MyDrive/Intro_ML_Summative/figures"
  os.makedirs(FIG_DIR, exist_ok=True)
  os.makedirs(MODEL_DIR, exist_ok=True)
except Exception:
  RESULTS_PATH = "/content/diabetes_results.csv"

Mounted at /content/drive


In [ ]:
file_id = "18K8zjp3Hc9scwP4_73BnxP7pKT9hsTC_"
zip_path = "/content/diabetes.zip"
extract_path = "/content/data"

os.makedirs(extract_path, exist_ok=True)

# Download using the file ID directly
gdown.download(id=file_id, output=zip_path, quiet=False)

# Extract the archive
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

# Load the dataset
df = pd.read_csv(f"{extract_path}/diabetic_data.csv", na_values="?")

Downloading...
From: https://drive.google.com/uc?id=18K8zjp3Hc9scwP4_73BnxP7pKT9hsTC_
To: /content/diabetes.zip
100%|██████████| 3.17M/3.17M [00:00<00:00, 21.0MB/s]
/tmp/ipykernel_2352/3317228282.py:15: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{extract_path}/diabetic_data.csv", na_values="?")


In [ ]:
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      99493 non-null   object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    3197 non-null    object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                61510 non-null   object
 11  medical_specialty         51817 non-null   object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

Checking the percentage of missing values in the dataset by columns.

In [ ]:
print("Raw shape:", df.shape)
print(df["readmitted"].value_counts())
print(df["readmitted"].value_counts(normalize=True).round(3))

# Missing values (the '?' entries are now NaN)
missing = df.isna().mean().sort_values(ascending=False)
print(missing[missing > 0].round(3))

# Duplicate patients
print("Unique patients:", df["patient_nbr"].nunique())

Raw shape: (101766, 50)
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64
readmitted
NO     0.539
>30    0.349
<30    0.112
Name: proportion, dtype: float64
weight               0.969
max_glu_serum        0.947
A1Cresult            0.833
medical_specialty    0.491
payer_code           0.396
race                 0.022
diag_3               0.014
diag_2               0.004
diag_1               0.000
dtype: float64
Unique patients: 71518


Checking the unique values in some columns to see what the dataset is like. Also noticed that some fields that had "?" got changed to Na values. Checking gender to make sure there arent any other entries beside male/ female and "glyburide" just to see the unique values the dataset operates by as this should apply to other fields like "migitol", "glucose".

In [ ]:
for col in ["gender", "race", "glyburide"]:
  print(df[col].value_counts(normalize=True, dropna=False) * 100)

for col in ['number_outpatient', 'number_emergency', 'number_inpatient']:
  print(df[col].max())

gender
Female             53.758623
Male               46.238429
Unknown/Invalid     0.002948
Name: proportion, dtype: float64
race
Caucasian          74.778413
AfricanAmerican    18.876639
NaN                 2.233555
Hispanic            2.001651
Other               1.479866
Asian               0.629876
Name: proportion, dtype: float64
glyburide
No        89.534815
Steady     9.113063
Up         0.797909
Down       0.554213
Name: proportion, dtype: float64
42
76
21


Fill back the mislead Na values as the string "None" or "?" are valid identifiers in some columns and should be renamed to show that.
Dropped weight as it is 96% null, encounter id as it isnt an important feature for the purpose of this model and payer code as that largely relates to the insurance coverage which wont apply every where in the world, and dropped invalid genders as their percentage is very small (0.003%).
Dropped patients who died as readmitted chance is 0%

In [ ]:
for col in ["A1Cresult", "max_glu_serum"]:
    df[col] = df[col].fillna("Not measured")

for col in ["medical_specialty", "race", "diag_1", "diag_2", "diag_3"]:
    df[col] = df[col].fillna("Unknown")

df = df.drop(columns=["weight", "encounter_id", "payer_code"])

# drop patients who died or discharged to hospice as they cant be readmitted
expired_codes = [11, 13, 14, 19, 20, 21]
df = df[~df["discharge_disposition_id"].isin(expired_codes)]

# drop invalid gender rows
df = df[df["gender"] != "Unknown/Invalid"]

print("Shape after cleaning:", df.shape)
print("Remaining NaNs:", df.isna().sum().sum())
print(df["readmitted"].value_counts(normalize=True).round(3))
print("Unique patients:", df["patient_nbr"].nunique())

Shape after cleaning: (99340, 47)
Remaining NaNs: 0
readmitted
NO     0.529
>30    0.357
<30    0.114
Name: proportion, dtype: float64
Unique patients: 69987


Drop column that have just 1 value as they offer no meaningful feature to learn from.

In [ ]:
const_cols = [c for c in df.columns if df[c].nunique() == 1]
print("Constant columns:", const_cols)
df = df.drop(columns=const_cols)

Constant columns: ['examide', 'citoglipton']


In [ ]:
def map_diag(code):
    if code == "Unknown":
        return "Unknown"
    if code.startswith(("V", "E")):
        return "Other"
    n = float(code)
    if 250 <= n < 251:
      return "Diabetes"
    elif 390 <= n <= 459 or n == 785:
      return "Circulatory"
    elif 460 <= n <= 519 or n == 786:
      return "Respiratory"
    elif 520 <= n <= 579 or n == 787:
      return "Digestive"
    elif 580 <= n <= 629 or n == 788:
      return "Genitourinary"
    elif 800 <= n <= 999:
      return "Injury"
    elif 710 <= n <= 739:
      return "Musculoskeletal"
    elif 140 <= n <= 239:
      return "Neoplasms"
    else:
      return "Other"

I grouped range of diag values into 9 groups following Strack et al grouping. Changed the id columns from int to str as they point to unique categories (IDS_mapping.csv). Age was set to get the mean/ midpoint of the range as so make it a quantity value. Then split the data set into train, val and test (70/15/15) and ensuring same patients stay together to reduce future overfitting.

In [ ]:
# group diag columns into 9 categories
for col in ["diag_1", "diag_2", "diag_3"]:
    df[col] = df[col].astype(str).map(map_diag)

# age bracket to mean
df["age"] = df["age"].str.extract(r"\[(\d+)-(\d+)\)").astype(int).mean(axis=1)

# ID columns are categories, not quantities
for col in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
    df[col] = df[col].astype(str)

# patient-level split: 70 / 15 / 15
from sklearn.model_selection import GroupShuffleSplit

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss1.split(df, groups=df["patient_nbr"]))
train, temp = df.iloc[train_idx], df.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss2.split(temp, groups=temp["patient_nbr"]))
val, test = temp.iloc[val_idx], temp.iloc[test_idx]

for name, part in [("train", train), ("val", val), ("test", test)]:
    print(name, part.shape, "patients:", part["patient_nbr"].nunique())
    print(part["readmitted"].value_counts(normalize=True).round(3).to_dict())

# no patient in more than one split
assert not set(train["patient_nbr"]) & set(val["patient_nbr"])
assert not set(train["patient_nbr"]) & set(test["patient_nbr"])
assert not set(val["patient_nbr"]) & set(test["patient_nbr"])

train (69613, 45) patients: 48990
{'NO': 0.528, '>30': 0.358, '<30': 0.114}
val (14868, 45) patients: 10498
{'NO': 0.53, '>30': 0.355, '<30': 0.116}
test (14859, 45) patients: 10499
{'NO': 0.531, '>30': 0.356, '<30': 0.113}


In [ ]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
from sklearn.metrics import (f1_score, accuracy_score, recall_score,
                             roc_auc_score, confusion_matrix, roc_curve, auc)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit

In [ ]:
label_map = {"NO": 0, ">30": 1, "<30": 2}
def split_xy(part):
    X = part.drop(columns=["readmitted", "patient_nbr"])
    y = part["readmitted"].map(label_map).to_numpy()
    groups = part["patient_nbr"].to_numpy()
    return X, y, groups

split each train, val, test set into features, labels and groups. We keep the patient id for future use in 5-fold cv. Then divide the columns into numeric, count, and categories.

In [ ]:
X_train, y_train, g_train = split_xy(train)
X_val, y_val, g_val = split_xy(val)
X_test, y_test, g_test = split_xy(test)

count_cols = ["number_outpatient", "number_emergency", "number_inpatient"]
num_cols = [c for c in X_train.select_dtypes(include="number").columns if c not in count_cols]
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()
print(len(num_cols), "numeric,", len(count_cols), "count,", len(cat_cols), "categorical")

6 numeric, 3 count, 34 categorical


Here we standardize or numeric columns and onehotencode our catergorical columns but for count columns we reduce them before standardizing as it has a huge outlier of 76 in the mix of the values and normal standardization wont resolve that when most values are 0-2

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("count", Pipeline([
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler()),
    ]), count_cols),
    ("cat", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=50, sparse_output=False), cat_cols),
])

# fit on TRAIN only, then apply to val and test
X_train_p = preprocessor.fit_transform(X_train).astype("float32")
X_val_p   = preprocessor.transform(X_val).astype("float32")
X_test_p  = preprocessor.transform(X_test).astype("float32")

print("Shapes:", X_train_p.shape, X_val_p.shape, X_test_p.shape)
print("NaNs in train:", np.isnan(X_train_p).sum())
print("Numeric col means on train (should be ~0):",
      X_train_p[:, :len(num_cols)].mean(axis=0).round(2))

# class weights for LogReg / RF / MLP
classes = np.array([0, 1, 2])
cw = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight = dict(zip(classes, cw))
print("Class weights:", {k: round(v, 2) for k, v in class_weight.items()})

Shapes: (69613, 185) (14868, 185) (14859, 185)
NaNs in train: 0
Numeric col means on train (should be ~0): [-0.  0.  0.  0. -0. -0.]
Class weights: {np.int64(0): np.float64(0.63), np.int64(1): np.float64(0.93), np.int64(2): np.float64(2.93)}


In [ ]:
import warnings
warnings.filterwarnings("ignore")

WANDB_ENABLED = False
WANDB_PROJECT = "summative-introtoML-kelvinaaron"

try:
    import wandb
    logged_in = False

    # --- Uncomment the block matching your environment, then set logged_in = True ---

    # Google Colab:
    from google.colab import userdata
    wandb.login(key=userdata.get("WANDB_API_KEY"))
    logged_in = True

    # Kaggle Notebooks:
    # from kaggle_secrets import UserSecretsClient
    # wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    # logged_in = True

    WANDB_ENABLED = logged_in
    if WANDB_ENABLED:
        print("W&B ready. Runs will be logged to project:", WANDB_PROJECT)
    else:
        print("W&B installed but not logged in yet — uncomment the block above for your environment.")
        print("Until then, the notebook still runs, but NOTHING is being tracked.")
except Exception as e:
    print("W&B not available — runs will NOT be logged until you fix this.")
    print("Reason:", e)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: k-aaron-onu (k-aaron-onu-african-leadership-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B ready. Runs will be logged to project: summative-introtoML-kelvinaaron


Created the reusable logger function and tracking macro f1 score to penalize disregarding of minority classes, accruacy to see hw well the model is performing on the entire dataset, roc_auc and recall for each class.

In [ ]:
RANDOM_STATE = 42
CLASS_NAMES = ["NO", ">30", "<30"]
results_log = []

def score_split(y_true, probs):
    preds = probs.argmax(axis=1)
    rec = recall_score(y_true, preds, labels=[0, 1, 2], average=None)
    return {
        "macro_f1": f1_score(y_true, preds, average="macro"),
        "accuracy": accuracy_score(y_true, preds),
        "roc_auc_ovr": roc_auc_score(y_true, probs, multi_class="ovr", average="macro"),
        "recall_NO":   rec[0],
        "recall_gt30": rec[1],
        "recall_lt30": rec[2],
    }

def log_experiment(run_name, config, y_train, train_probs, y_val, val_probs):
    # float32 softmax outputs can fail sklearn's "sums to 1" check, so renormalise
    train_probs = train_probs.astype("float64"); train_probs /= train_probs.sum(axis=1, keepdims=True)
    val_probs = val_probs.astype("float64");   val_probs   /= val_probs.sum(axis=1, keepdims=True)

    tr = score_split(y_train, train_probs)
    va = score_split(y_val, val_probs)
    print(f"[{run_name}] train macro-F1 = {tr['macro_f1']:.4f} | val macro-F1 = {va['macro_f1']:.4f} | "
          f"val recall(<30) = {va['recall_lt30']:.4f} | val AUC = {va['roc_auc_ovr']:.4f}")

    cm = confusion_matrix(y_val, val_probs.argmax(axis=1), labels=[0, 1, 2])

    if WANDB_ENABLED:
        try:
            run = wandb.init(project=WANDB_PROJECT, name=run_name, config=config, reinit=True)
            log_dict = {**{f"train_{k}": v for k, v in tr.items()},
                        **{f"val_{k}": v for k, v in va.items()}}
            fig, ax = plt.subplots(figsize=(4, 4))
            ax.imshow(cm, cmap="Blues")
            for i in range(3):
                for j in range(3):
                    ax.text(j, i, str(cm[i, j]), ha="center", va="center")
            ax.set_xticks(range(3)); ax.set_xticklabels(CLASS_NAMES)
            ax.set_yticks(range(3)); ax.set_yticklabels(CLASS_NAMES)
            ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(run_name)
            wandb.log({**log_dict, "confusion_matrix": wandb.Image(fig)})
            plt.close(fig)
            run.finish()
        except Exception as e:
            print(f"  (W&B logging failed for this run: {e} — local results still recorded)")

    results_log.append({
        "run_name": run_name,
        **config,
        **{f"train_{k}": round(v, 4) for k, v in tr.items()},
        **{f"val_{k}": round(v, 4) for k, v in va.items()},
    })

Ran a logistic regression baseline using SAGA as it works well with large datasets and can act as a form of early stopping when change in weights gets equal to the tolerance

In [ ]:
logreg = LogisticRegression(C=1, penalty="l2", solver="saga", max_iter=1000,
                            class_weight="balanced", random_state=RANDOM_STATE)
logreg.fit(X_train_p, y_train)

log_experiment(
    run_name="logreg_C1_l2",
    config={"model": "LogisticRegression", "C": 1, "penalty": "l2", "solver": "saga",
            "max_iter": 1000, "class_weight": "balanced", "random_state": 42},
    y_train=y_train,
    train_probs=logreg.predict_proba(X_train_p),
    y_val=y_val,
    val_probs=logreg.predict_proba(X_val_p),
)

[logreg_C1_l2] train macro-F1 = 0.4403 | val macro-F1 = 0.4344 | val recall(<30) = 0.4191 | val AUC = 0.6560


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


train_accuracy,▁
train_macro_f1,▁
train_recall_NO,▁
train_recall_gt30,▁
train_recall_lt30,▁
train_roc_auc_ovr,▁
val_accuracy,▁
val_macro_f1,▁
val_recall_NO,▁
val_recall_gt30,▁
+2,...


The test and validation f1 scores are close showing little to no overfitting of the model. Recall for "<30" is a little low with 43 and 41 for train and val respectively.
This is probably showing the limits of what a linear model can achieve.

In [ ]:
rf_params = {
    "n_estimators": 300,
    "max_depth": 15,
    "min_samples_leaf": 5,
    "max_features": "sqrt",
    "class_weight": "balanced_subsample",
    "n_jobs": -1,
    "random_state": 42,
}
rf = RandomForestClassifier(**rf_params)
rf.fit(X_train_p, y_train)

log_experiment(
    run_name="rf_base_d15_leaf5",
    config={"model": "RandomForest", **rf_params},
    y_train=y_train,
    train_probs=rf.predict_proba(X_train_p),
    y_val=y_val,
    val_probs=rf.predict_proba(X_val_p),
)

[rf_base_d15_leaf5] train macro-F1 = 0.6020 | val macro-F1 = 0.4529 | val recall(<30) = 0.3283 | val AUC = 0.6644


train_accuracy,▁
train_macro_f1,▁
train_recall_NO,▁
train_recall_gt30,▁
train_recall_lt30,▁
train_roc_auc_ovr,▁
val_accuracy,▁
val_macro_f1,▁
val_recall_NO,▁
val_recall_gt30,▁
+2,...


the model seems to be slightly overfitting as train f1 (60%) is much higher than validation (45%). And also a reduction in recall (0.328 against 0.419) from the logistic regression model. This baseline memorized more of the dataset than finding signals. For the next experiment we will reduce the depth to 10 and increase min_sample_leaf to 20 to combat the overfitting

In [ ]:
rf_params_2 = {
    "n_estimators": 300,
    "max_depth": 10,
    "min_samples_leaf": 20,
    "max_features": "sqrt",
    "class_weight": "balanced_subsample",
    "n_jobs": -1,
    "random_state": 42,
}
rf2 = RandomForestClassifier(**rf_params_2)
rf2.fit(X_train_p, y_train)

log_experiment(
    run_name="rf_d10_leaf20",
    config={"model": "RandomForest", **rf_params_2},
    y_train=y_train,
    train_probs=rf2.predict_proba(X_train_p),
    y_val=y_val,
    val_probs=rf2.predict_proba(X_val_p),
)

[rf_d10_leaf20] train macro-F1 = 0.4616 | val macro-F1 = 0.4252 | val recall(<30) = 0.4430 | val AUC = 0.6599


train_accuracy,▁
train_macro_f1,▁
train_recall_NO,▁
train_recall_gt30,▁
train_recall_lt30,▁
train_roc_auc_ovr,▁
val_accuracy,▁
val_macro_f1,▁
val_recall_NO,▁
val_recall_gt30,▁
+2,...


The overfitting gap closed, from 0.149 (0.602 against 0.453) to 0.036 (0.462 against 0.425), but validation macro-F1 fell from 0.453 to 0.425, meaning less balance between the precision and recall for this model but less overfitting. Validation AUC barely moved (0.664 to 0.660), so the ranking quality of both forests is similar. The shallower forest catches more <30 patients (recall 0.443 against 0.328) but fewer >30 patients (0.296 against 0.420), and losing more on >30 than it gains on <30 shows why macro-F1 went down by alot. Lets try a new model in xgboost as it before relatively befores alot better than random forests especially when properly tuned and on larger datasets like this one.

In [ ]:
xgb_params = {
    "objective": "multi:softprob",
    "n_estimators": 400,
    "learning_rate": 0.1,
    "max_depth": 5,
    "min_child_weight": 1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "tree_method": "hist",
    "eval_metric": "mlogloss",
    "n_jobs": -1,
    "random_state": 42,
}
xgb = XGBClassifier(**xgb_params)
sw_train = compute_sample_weight("balanced", y_train)
xgb.fit(X_train_p, y_train, sample_weight=sw_train)

log_experiment(
    run_name="xgb_base_d5_lr0.1",
    config={"model": "XGBoost", **xgb_params, "sample_weight": "balanced"},
    y_train=y_train,
    train_probs=xgb.predict_proba(X_train_p),
    y_val=y_val,
    val_probs=xgb.predict_proba(X_val_p),
)

[xgb_base_d5_lr0.1] train macro-F1 = 0.5524 | val macro-F1 = 0.4525 | val recall(<30) = 0.3760 | val AUC = 0.6652


train_accuracy,▁
train_macro_f1,▁
train_recall_NO,▁
train_recall_gt30,▁
train_recall_lt30,▁
train_roc_auc_ovr,▁
val_accuracy,▁
val_macro_f1,▁
val_recall_NO,▁
val_recall_gt30,▁
+2,...


The baseline xgbboost model seems to be moderately overfitting with all train values having a sizable gap to validation. But still it gave the highest val AUC score seen so far (0.665) so an improvement in that regard over previous models.

In [ ]:
xgb_params_2 = {
    "objective": "multi:softprob",
    "n_estimators": 400,
    "learning_rate": 0.05,
    "max_depth": 3,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.5,
    "reg_lambda": 5,
    "tree_method": "hist",
    "eval_metric": "mlogloss",
    "n_jobs": -1,
    "random_state": 42,
}
xgb2 = XGBClassifier(**xgb_params_2)
sw_train = compute_sample_weight("balanced", y_train)
xgb2.fit(X_train_p, y_train, sample_weight=sw_train)

log_experiment(
    run_name="xgb_d3_lr0.05_mcw10",
    config={"model": "XGBoost", **xgb_params_2, "sample_weight": "balanced"},
    y_train=y_train,
    train_probs=xgb2.predict_proba(X_train_p),
    y_val=y_val,
    val_probs=xgb2.predict_proba(X_val_p),
)

[xgb_d3_lr0.05_mcw10] train macro-F1 = 0.4566 | val macro-F1 = 0.4438 | val recall(<30) = 0.4144 | val AUC = 0.6649


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


train_accuracy,▁
train_macro_f1,▁
train_recall_NO,▁
train_recall_gt30,▁
train_recall_lt30,▁
train_roc_auc_ovr,▁
val_accuracy,▁
val_macro_f1,▁
val_recall_NO,▁
val_recall_gt30,▁
+2,...


This XGBoost run matches the baseline’s ranking quality with far less overfitting. Its train/validation gap is 0.013, compared with 0.100 for the baseline, and validation AUC is 0.6649 against 0.6652. Validation macro-F1 is slightly lower (0.444 against 0.453), but <30 recall is higher (0.414 against 0.376). The major gain will be a reduction in overfitting while keeping quality.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

In [ ]:
def build_mlp(n_features, p):
    model = keras.Sequential([layers.Input(shape=(n_features,))])
    for u in p["hidden_units"]:
        model.add(layers.Dense(u, activation=p["activation"],
                               kernel_regularizer=regularizers.l2(p["l2"])))
        if p["batch_norm"]:
            model.add(layers.BatchNormalization())
        model.add(layers.Dropout(p["dropout"]))
    model.add(layers.Dense(3, activation="softmax"))
    model.compile(optimizer=keras.optimizers.Adam(p["learning_rate"]),
                  loss="sparse_categorical_crossentropy")
    return model

In [ ]:
mlp_params = {
    "hidden_units": [128, 64],
    "activation": "relu",
    "dropout": 0.3,
    "l2": 1e-4,
    "batch_norm": False,
    "learning_rate": 1e-3,
    "batch_size": 512,
    "max_epochs": 100,
    "early_stopping_patience": 8,
    "class_weight": "balanced",
    "random_state": 42,
}

tf.keras.utils.set_random_seed(mlp_params["random_state"])
mlp = build_mlp(X_train_p.shape[1], mlp_params)
mlp.summary()

early = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=mlp_params["early_stopping_patience"],
    restore_best_weights=True
    )
cw = {int(k): float(v) for k, v in class_weight.items()}

history = mlp.fit(
    X_train_p, y_train,
    validation_data=(X_val_p, y_val),
    epochs=mlp_params["max_epochs"],
    batch_size=mlp_params["batch_size"],
    class_weight=cw, callbacks=[early], verbose=2
    )

log_experiment(
    run_name="mlp_128-64_do0.3_l2e-4",
    config={"model": "MLP", **mlp_params, "epochs_run": len(history.history["loss"])},
    y_train=y_train,
    train_probs=mlp.predict(X_train_p, batch_size=1024),
    y_val=y_val,
    val_probs=mlp.predict(X_val_p, batch_size=1024),
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        23,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,259 (126.01 KB)

 Trainable params: 32,259 (126.01 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
136/136 - 7s - 49ms/step - loss: 1.0906 - val_loss: 1.0574
Epoch 2/100
136/136 - 3s - 24ms/step - loss: 1.0656 - val_loss: 1.0427
Epoch 3/100
136/136 - 3s - 19ms/step - loss: 1.0575 - val_loss: 1.0365
Epoch 4/100
136/136 - 5s - 38ms/step - loss: 1.0511 - val_loss: 1.0316
Epoch 5/100
136/136 - 2s - 14ms/step - loss: 1.0449 - val_loss: 1.0281
Epoch 6/100
136/136 - 2s - 16ms/step - loss: 1.0419 - val_loss: 1.0291
Epoch 7/100
136/136 - 2s - 14ms/step - loss: 1.0378 - val_loss: 1.0230
Epoch 8/100
136/136 - 1s - 8ms/step - loss: 1.0356 - val_loss: 1.0356
Epoch 9/100
136/136 - 1s - 8ms/step - loss: 1.0330 - val_loss: 1.0304
Epoch 10/100
136/136 - 1s - 8ms/step - loss: 1.0295 - val_loss: 1.0331
Epoch 11/100
136/136 - 1s - 8ms/step - loss: 1.0282 - val_loss: 1.0377
Epoch 12/100
136/136 - 1s - 8ms/step - loss: 1.0265 - val_loss: 1.0206
Epoch 13/100
136/136 - 1s - 8ms/step - loss: 1.0232 - val_loss: 1.0230
Epoch 14/100
136/136 - 1s - 9ms/step - loss: 1.0200 - val_loss: 1.0293
Epoch 15

train_accuracy,▁
train_macro_f1,▁
train_recall_NO,▁
train_recall_gt30,▁
train_recall_lt30,▁
train_roc_auc_ovr,▁
val_accuracy,▁
val_macro_f1,▁
val_recall_NO,▁
val_recall_gt30,▁
+2,...


The baseline MLP lands in the same range as the other models without beating them. Maybe some what worse than the tuned xgboost model. It showed slight overfitting between train and val values as seen in f1 and auc score so lets try to see the gains of reducing that overfitting.

In [ ]:
mlp_params_2 = {
    "hidden_units": [64, 32],
    "activation": "relu",
    "dropout": 0.5,
    "l2": 1e-3,
    "batch_norm": False,
    "learning_rate": 1e-3,
    "batch_size": 512,
    "max_epochs": 100,
    "early_stopping_patience": 8,
    "class_weight": "balanced",
    "random_state": 42,
}

tf.keras.utils.set_random_seed(mlp_params_2["random_state"])
mlp2 = build_mlp(X_train_p.shape[1], mlp_params_2)
cw = {int(k): float(v) for k, v in class_weight.items()}

early = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=mlp_params_2["early_stopping_patience"],
    restore_best_weights=True)
history2 = mlp2.fit(
    X_train_p, y_train,
    validation_data=(X_val_p, y_val),
    epochs=mlp_params_2["max_epochs"],
    batch_size=mlp_params_2["batch_size"],
    class_weight=cw, callbacks=[early], verbose=2)

log_experiment(
    run_name="mlp_64-32_do0.5_l2e-3",
    config={"model": "MLP", **mlp_params_2, "epochs_run": len(history2.history["loss"])},
    y_train=y_train,
    train_probs=mlp2.predict(X_train_p, batch_size=1024),
    y_val=y_val,
    val_probs=mlp2.predict(X_val_p, batch_size=1024),
)

Epoch 1/100
136/136 - 7s - 53ms/step - loss: 1.2117 - val_loss: 1.1545
Epoch 2/100
136/136 - 2s - 12ms/step - loss: 1.1312 - val_loss: 1.1039
Epoch 3/100
136/136 - 2s - 13ms/step - loss: 1.1021 - val_loss: 1.0820
Epoch 4/100
136/136 - 1s - 6ms/step - loss: 1.0876 - val_loss: 1.0646
Epoch 5/100
136/136 - 1s - 6ms/step - loss: 1.0770 - val_loss: 1.0495
Epoch 6/100
136/136 - 1s - 11ms/step - loss: 1.0701 - val_loss: 1.0457
Epoch 7/100
136/136 - 1s - 10ms/step - loss: 1.0669 - val_loss: 1.0556
Epoch 8/100
136/136 - 2s - 14ms/step - loss: 1.0627 - val_loss: 1.0337
Epoch 9/100
136/136 - 2s - 18ms/step - loss: 1.0588 - val_loss: 1.0334
Epoch 10/100
136/136 - 2s - 18ms/step - loss: 1.0556 - val_loss: 1.0317
Epoch 11/100
136/136 - 1s - 5ms/step - loss: 1.0543 - val_loss: 1.0359
Epoch 12/100
136/136 - 1s - 5ms/step - loss: 1.0524 - val_loss: 1.0296
Epoch 13/100
136/136 - 1s - 6ms/step - loss: 1.0504 - val_loss: 1.0376
Epoch 14/100
136/136 - 1s - 6ms/step - loss: 1.0510 - val_loss: 1.0382
Epoch 1

train_accuracy,▁
train_macro_f1,▁
train_recall_NO,▁
train_recall_gt30,▁
train_recall_lt30,▁
train_roc_auc_ovr,▁
val_accuracy,▁
val_macro_f1,▁
val_recall_NO,▁
val_recall_gt30,▁
+2,...


The extra regularization worked as intended to reduce overfitting Train and validation macro-F1 are now 0.450 and 0.442, a gap of 0.008 against 0.053 for the first network. Validation AUC rose from 0.658 to 0.663, and validation macro-F1 stayed essentially flat (0.4415 to 0.4422). <30 recall did fall from 0.414 to 0.385.

Considering all experiments so far every model fell in a narrow band with validation macro-F1 between 0.425 and 0.453 and AUC between 0.656 and 0.665. The main finding for the report is that deep learning, boosting and bagging all reach roughly the same ceiling on these features, so the limit maybe in the data and not the model.

In [ ]:
xgb_fixed = {
    "objective": "multi:softprob",
    "tree_method": "hist",
    "eval_metric": "mlogloss",
    "n_jobs": -1,
    "random_state": 42,
}
xgb_search_space = {
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4, 5, 6],
    "n_estimators": [200, 400, 600],
    "min_child_weight": [1, 5, 10, 20],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.4, 0.6, 0.8],
    "reg_lambda": [1, 5, 10],
}

# train rows = fold -1 (always training), validation rows = fold 0 (the one test fold)
X_all = np.vstack([X_train_p, X_val_p])
y_all = np.concatenate([y_train, y_val])
test_fold = np.concatenate([-np.ones(len(y_train)), np.zeros(len(y_val))])
sw_all = np.concatenate([compute_sample_weight("balanced", y_train), np.ones(len(y_val))])

search = RandomizedSearchCV(
    XGBClassifier(**xgb_fixed),
    param_distributions=xgb_search_space,
    n_iter=20,
    scoring={"macro_f1": "f1_macro", "auc": "roc_auc_ovr"},
    refit=False,
    cv=PredefinedSplit(test_fold),
    random_state=42,
    n_jobs=1,
    verbose=2,
)
search.fit(X_all, y_all, sample_weight=sw_all)

trials = pd.DataFrame(search.cv_results_)
cols = ["mean_test_macro_f1", "mean_test_auc"] + [f"param_{k}" for k in xgb_search_space]
trials = trials[cols].sort_values("mean_test_macro_f1", ascending=False)
trials.to_csv("/content/drive/MyDrive/Intro_ML_Summative/results/xgb_search_trials.csv", index=False)
print(trials.head(10).round(4).to_string(index=False))

Fitting 1 folds for each of 20 candidates, totalling 20 fits
[CV] END colsample_bytree=0.4, learning_rate=0.05, max_depth=4, min_child_weight=20, n_estimators=600, reg_lambda=5, subsample=1.0; total time=  55.0s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=6, min_child_weight=20, n_estimators=600, reg_lambda=1, subsample=0.8; total time= 1.3min
[CV] END colsample_bytree=0.6, learning_rate=0.1, max_depth=5, min_child_weight=10, n_estimators=400, reg_lambda=5, subsample=1.0; total time=  42.4s
[CV] END colsample_bytree=0.4, learning_rate=0.03, max_depth=6, min_child_weight=5, n_estimators=200, reg_lambda=10, subsample=0.8; total time=  30.1s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=2, min_child_weight=20, n_estimators=600, reg_lambda=10, subsample=0.8; total time=  48.9s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=3, min_child_weight=20, n_estimators=400, reg_lambda=10, subsample=0.6; total time=  37.9s
[CV] END colsample_bytree=0.6, le

All seven tracked runs and the top ten search trials fall in validation AUC 0.656–0.669, across linear, bagged, boosted and neural models. This is consistent with a data ceiling limitation pointed out earlier.

In [ ]:
def save_results():
    pd.DataFrame(results_log).to_csv(RESULTS_PATH, index=False)

# after a reset, reload earlier runs before logging new ones
if os.path.exists(RESULTS_PATH):
    results_log = pd.read_csv(RESULTS_PATH).to_dict("records")

In [ ]:
save_results()

In [ ]:
joblib.dump(xgb2, f"{MODEL_DIR}/best_xgb_d3_lr0.05.joblib")
mlp2.save(f"{MODEL_DIR}/best_mlp_64-32.h5")
print(os.listdir(MODEL_DIR))

['best_xgb_d3_lr0.05.joblib', 'best_mlp_64-32.h5']


In [ ]:
def norm(p):
    p = p.astype("float64")
    return p / p.sum(axis=1, keepdims=True)

test_scores = {
    "xgb_d3_lr0.05": score_split(y_test, norm(xgb2.predict_proba(X_test_p))),
    "mlp_64-32": score_split(y_test, norm(mlp2.predict(X_test_p, batch_size=1024))),
}
print(pd.DataFrame(test_scores).T.round(4))

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
               macro_f1  accuracy  roc_auc_ovr  recall_NO  recall_gt30  \
xgb_d3_lr0.05    0.4462    0.5065       0.6695     0.6104        0.375   
mlp_64-32        0.2735    0.4654       0.4845     0.8191        0.064   

               recall_lt30  
xgb_d3_lr0.05       0.4337  
mlp_64-32           0.0708  


In [ ]:
from sklearn.preprocessing import label_binarize

plt.rcParams.update({"font.family": "serif", "font.size": 10})

test_probs = {
    "xgb_d3_lr0.05_mcw10":   norm(xgb2.predict_proba(X_test_p)),
    "mlp_64-32_do0.5_l2e-3": norm(mlp2.predict(X_test_p, batch_size=1024)),
}

# Confusion matrices (row-normalised, counts in brackets), test set
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4))
for ax, (name, probs), tag in zip(axes, test_probs.items(), "ab"):
    cm = confusion_matrix(y_test, probs.argmax(axis=1), labels=[0, 1, 2])
    cmn = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cmn[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center",
                    color="white" if cmn[i, j] > 0.6 else "black", fontsize=8)
    ax.set_xticks(range(3)); ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticks(range(3)); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(f"({tag}) {name}", fontsize=9)
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/fig_confusion_matrices.png", dpi=300, bbox_inches="tight"); plt.close(fig)

# One-vs-rest ROC curves, test set
y_bin = label_binarize(y_test, classes=[0, 1, 2])
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4))
for ax, (name, probs), tag in zip(axes, test_probs.items(), "ab"):
    for k, cname in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_bin[:, k], probs[:, k])
        ax.plot(fpr, tpr, label=f"{cname} (AUC = {auc(fpr, tpr):.3f})")
    ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=0.8)
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title(f"({tag}) {name}", fontsize=9); ax.legend(loc="lower right", fontsize=8)
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/fig_roc_curves.png", dpi=300, bbox_inches="tight"); plt.close(fig)

# Learning curves: XGBoost per boosting round, MLPs per epoch
xgb_curve = XGBClassifier(**xgb_params_2)
xgb_curve.fit(X_train_p, y_train, sample_weight=sw_train,
              eval_set=[(X_train_p, y_train), (X_val_p, y_val)], verbose=False)
ev = xgb_curve.evals_result()
panels = [("xgb_d3_lr0.05_mcw10", ev["validation_0"]["mlogloss"], ev["validation_1"]["mlogloss"], "Boosting round", "Log loss")]
if "history" in globals():
    panels.append(("mlp_128-64_do0.3_l2e-4", history.history["loss"], history.history["val_loss"], "Epoch", "Loss"))
panels.append(("mlp_64-32_do0.5_l2e-3", history2.history["loss"], history2.history["val_loss"], "Epoch", "Loss"))

fig, axes = plt.subplots(1, len(panels), figsize=(2.4 * len(panels) + 0.6, 3.2))
for ax, (name, tr, va, xl, yl), tag in zip(np.atleast_1d(axes), panels, "abc"):
    ax.plot(tr, label="Train"); ax.plot(va, label="Validation")
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.set_title(f"({tag}) {name}", fontsize=8); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/fig_learning_curves.png", dpi=300, bbox_inches="tight"); plt.close(fig)

print(os.listdir(FIG_DIR))

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
['fig_confusion_matrices.png', 'fig_roc_curves.png', 'fig_learning_curves.png']
